# 05  DQN Agent Training
---

## What this notebook does

This notebook trains a **Deep Q-Network (DQN)** agent to learn an optimal
task scheduling policy through reinforcement learning.

The agent interacts with the `TaskSchedulerEnv` built in notebook 04,
playing 5,000 episodes of the scheduling game and gradually learning
which tasks to prioritise in order to maximise on-time completions.

### DQN Architecture
```
Input (140)  →  Dense(256, ReLU)  →  Dense(256, ReLU)  →  Dense(128, ReLU)  →  Output(20)
```

### Key training components
| Component | Value | Purpose |
|---|---|---|
| Experience replay buffer | 10,000 transitions | Breaks correlation between consecutive samples |
| Batch size | 64 | Stable gradient estimates |
| Epsilon (start→end) | 1.0 → 0.05 | Exploration → exploitation |
| Epsilon decay steps | 50,000 | Gradual transition |
| Learning rate | 0.001 | Adam optimiser |
| Target network update | Every 100 episodes | Stabilises training |
| Discount factor γ | 0.95 | From MDP formulation |
| Training episodes | 5,000 | Full training run |

### Notebook structure
1. Imports and configuration
2. Copy environment code (self-contained)
3. DQN neural network
4. Experience replay buffer
5. DQN agent
6. Training loop
7. Training results and charts
8. Save trained model
9. Quick evaluation on test set
10. Training summary


## 1. Imports and Configuration

In [ ]:
import json
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device          : {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_PATH   = "data/train_scenarios.json"
TEST_PATH    = "data/test_scenarios.json"
MODEL_PATH   = "data/dqn_model.pth"
RESULTS_PATH = "data/training_results.csv"

# ── Environment config ────────────────────────────────────────────────────────
WORKDAY_MINUTES = 600
MAX_TASKS       = 20
N_FEATURES      = 7
STATE_SIZE      = MAX_TASKS * N_FEATURES   # 140

# ── Reward values ─────────────────────────────────────────────────────────────
R_ON_TIME      = +10.0
R_MISSED       = -15.0
R_REORDER      = -2.0
R_DEP_RESOLVED = +5.0
R_VAGUE        = +3.0

# ── DQN hyperparameters ───────────────────────────────────────────────────────
HIDDEN_1        = 256
HIDDEN_2        = 256
HIDDEN_3        = 128
LEARNING_RATE   = 0.001
GAMMA           = 0.95
BUFFER_SIZE     = 10000
BATCH_SIZE      = 64
EPSILON_START   = 1.0
EPSILON_END     = 0.05
EPSILON_DECAY   = 50000
TARGET_UPDATE   = 100
N_EPISODES      = 5000
LOG_EVERY       = 100

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})
PALETTE = ["#1D9E75", "#534AB7", "#BA7517", "#2E75B6", "#D85A30"]

print(f"State size      : {STATE_SIZE}")
print(f"Action space    : {MAX_TASKS}")
print(f"Architecture    : {STATE_SIZE} -> {HIDDEN_1} -> {HIDDEN_2} -> {HIDDEN_3} -> {MAX_TASKS}")
print(f"Training ep.    : {N_EPISODES}")
print(f"Replay buffer   : {BUFFER_SIZE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Epsilon         : {EPSILON_START} -> {EPSILON_END} over {EPSILON_DECAY} steps")


## 2. Environment Code

The full environment is reproduced here so this notebook is self-contained.
This is the same `TaskSchedulerEnv` from notebook 04 with the safety timeout.


In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    train_scenarios = json.load(f)
with open(TEST_PATH) as f:
    test_scenarios = json.load(f)

print(f"Train scenarios : {len(train_scenarios)}")
print(f"Test scenarios  : {len(test_scenarios)}")

# ── State encoding ────────────────────────────────────────────────────────────
def encode_task(task, current_time, completed_ids):
    time_left        = max(task['deadline_min'] - current_time, 0)
    deadline_urgency = float(np.clip(1.0 - (time_left / WORKDAY_MINUTES), 0.0, 1.0))
    duration_norm    = float(np.clip(task['duration_min'] / 240.0, 0.0, 1.0))
    priority_norm    = (task['priority'] - 1) / 4.0
    unmet_deps       = [d for d in task.get('dependencies', []) if d not in completed_ids]
    has_dependency   = 1.0 if unmet_deps else 0.0
    is_vague         = 1.0 if task.get('vague', False) else 0.0
    time_pressure    = float(np.clip(task['duration_min'] / max(time_left, 1), 0.0, 1.0))
    is_available     = 1.0
    return np.array([deadline_urgency, duration_norm, priority_norm,
                     has_dependency, is_vague, time_pressure, is_available],
                    dtype=np.float32)

def encode_state(task_queue, current_time, completed_ids):
    state = np.zeros((MAX_TASKS, N_FEATURES), dtype=np.float32)
    for i, task in enumerate(task_queue[:MAX_TASKS]):
        state[i] = encode_task(task, current_time, completed_ids)
    return state.flatten()

def compute_reward(task, finish_time, prev_order, completed_ids):
    reward = 0.0
    info   = {}
    if finish_time <= task['deadline_min']:
        reward += R_VAGUE if task.get('vague', False) else R_ON_TIME
        info['on_time'] = True
    else:
        reward += R_MISSED
        info['on_time'] = False
    for dep_id in task.get('dependencies', []):
        if dep_id in completed_ids:
            reward += R_DEP_RESOLVED
    if prev_order and task['id'] != prev_order[0]:
        reward += R_REORDER
    info['reward'] = reward
    return reward, info

# ── Environment class ─────────────────────────────────────────────────────────
class TaskSchedulerEnv:
    def __init__(self, scenarios, workday_minutes=WORKDAY_MINUTES, seed=42):
        self.scenarios               = scenarios
        self.workday_minutes         = workday_minutes
        self.rng                     = random.Random(seed)
        self.observation_space_shape = (STATE_SIZE,)
        self.action_space_n          = MAX_TASKS
        self.task_queue = []; self.pending_disrupts = []
        self.current_time = 0; self.completed = []; self.missed = []
        self.episode_reward = 0.0; self.prev_order = []
        self.scenario = None; self.steps = 0

    def reset(self):
        self.scenario         = self.rng.choice(self.scenarios)
        self.task_queue       = copy.deepcopy(self.scenario['tasks'])
        self.pending_disrupts = copy.deepcopy(self.scenario['disruptions'])
        self.current_time     = 0; self.completed = []; self.missed = []
        self.episode_reward   = 0.0
        self.prev_order       = [t['id'] for t in self.task_queue]
        self.steps            = 0
        return self._get_state()

    def step(self, action):
        self.steps += 1
        if self.steps > 200 or self.current_time >= self.workday_minutes:
            self.missed.extend(self.task_queue); self.task_queue = []
            return self._get_state(), 0.0, True, {'reason': 'timeout'}
        self._apply_disruptions()
        if not self.task_queue:
            return self._get_state(), 0.0, True, {'reason': 'empty_queue'}
        action = int(min(action, len(self.task_queue) - 1))
        task   = self.task_queue.pop(action)
        completed_ids = {t['id'] for t in self.completed}
        if task.get('dependencies'):
            unmet = [d for d in task['dependencies'] if d not in completed_ids]
            if unmet:
                self.task_queue.append(task); self.current_time += 10
                return self._get_state(), -1.0, False, {'reason': 'unmet_dep'}
        finish_time = self.current_time + task['duration_min']
        if finish_time > self.workday_minutes:
            self.missed.append(task); self.episode_reward += R_MISSED
            done = len(self.task_queue) == 0
            return self._get_state(), R_MISSED, done, {'reason': 'no_time', 'on_time': False}
        self.current_time = finish_time
        completed_ids     = {t['id'] for t in self.completed}
        reward, info      = compute_reward(task, finish_time, self.prev_order, completed_ids)
        self.completed.append({**task, 'finish_time': finish_time, 'on_time': info.get('on_time', False)})
        self.episode_reward += reward
        self.prev_order      = [t['id'] for t in self.task_queue]
        done = (self.current_time >= self.workday_minutes) or (len(self.task_queue) == 0)
        if done and self.task_queue:
            self.missed.extend(self.task_queue); self.task_queue = []
        return self._get_state(), reward, done, info

    def _get_state(self):
        completed_ids = {t['id'] for t in self.completed}
        return encode_state(self.task_queue, self.current_time, completed_ids)

    def _apply_disruptions(self):
        remaining = []
        for d in self.pending_disrupts:
            if d['type'] == 'urgent_insertion':
                if self.current_time >= d.get('arrival_min', 0):
                    self.task_queue.append(copy.deepcopy(d['task']))
                else:
                    remaining.append(d)
            elif d['type'] == 'deadline_shift':
                for t in self.task_queue:
                    if t['id'] == d['target_task_id']:
                        t['deadline_min'] = d['new_deadline_min']
            elif d['type'] == 'task_cancellation':
                self.task_queue = [t for t in self.task_queue if t['id'] != d.get('target_task_id')]
        self.pending_disrupts = remaining

    def get_metrics(self):
        total   = len(self.completed) + len(self.missed)
        on_time = [t for t in self.completed if t.get('on_time')]
        tcr     = len(on_time) / total if total > 0 else 0.0
        devs    = [abs(t['finish_time'] - t['deadline_min']) for t in self.completed if 'finish_time' in t]
        dd      = float(np.mean(devs)) if devs else float(WORKDAY_MINUTES)
        return {'tcr': tcr, 'dd': dd, 'episode_reward': self.episode_reward,
                'completed': len(self.completed), 'missed': len(self.missed)}

    def action_mask(self):
        mask = np.zeros(MAX_TASKS, dtype=bool)
        mask[:len(self.task_queue)] = True
        return mask

print("Environment loaded successfully.")


## 3. DQN Neural Network

The DQN uses a 3-hidden-layer feedforward network.
It takes the 140-dimensional state vector as input and outputs
one Q-value per action (20 Q-values total).

The Q-value Q(s, a) represents the agent's estimate of the
total future discounted reward for taking action a in state s.


In [ ]:
class DQNetwork(nn.Module):
    """
    Deep Q-Network with 3 hidden layers.
    Input:  state vector of shape (STATE_SIZE,) = (140,)
    Output: Q-values of shape (MAX_TASKS,) = (20,)
    """

    def __init__(self, state_size=STATE_SIZE, action_size=MAX_TASKS,
                 h1=HIDDEN_1, h2=HIDDEN_2, h3=HIDDEN_3):
        super(DQNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, h3),
            nn.ReLU(),
            nn.Linear(h3, action_size),
        )

    def forward(self, x):
        return self.net(x)


# ── Print architecture ────────────────────────────────────────────────────────
net = DQNetwork().to(DEVICE)
print("DQN Architecture:")
print(net)
print()

total_params = sum(p.numel() for p in net.parameters())
trainable    = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Device              : {DEVICE}")


## 4. Experience Replay Buffer

The replay buffer stores past (state, action, reward, next_state, done)
transitions and samples random mini-batches for training.

**Why experience replay?**
Without it, the agent would train on consecutive transitions which are
highly correlated — the agent sees similar states one after another.
This leads to unstable training and poor generalisation.
Random sampling from the buffer breaks this correlation.


In [ ]:
class ReplayBuffer:
    """
    Fixed-size circular buffer storing experience tuples.
    Samples random mini-batches for DQN training.
    """

    def __init__(self, capacity=BUFFER_SIZE):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Store one transition."""
        self.buffer.append((
            np.array(state,      dtype=np.float32),
            int(action),
            float(reward),
            np.array(next_state, dtype=np.float32),
            bool(done),
        ))

    def sample(self, batch_size=BATCH_SIZE):
        """Sample a random mini-batch. Returns tensors on DEVICE."""
        batch  = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(DEVICE),
            torch.LongTensor(actions).to(DEVICE),
            torch.FloatTensor(rewards).to(DEVICE),
            torch.FloatTensor(np.array(next_states)).to(DEVICE),
            torch.FloatTensor(dones).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)


print(f"ReplayBuffer defined. Capacity: {BUFFER_SIZE:,} transitions.")
print(f"Minimum fill before training  : {BATCH_SIZE} transitions.")


## 5. DQN Agent

The DQN agent wraps the network and replay buffer.
It uses two networks:
- **Online network** — updated every training step
- **Target network** — copy of online network, updated every 100 episodes

The target network stabilises training by providing consistent
Q-value targets that do not change every step.


In [ ]:
class DQNAgent:
    """
    DQN agent with experience replay and target network.
    """

    def __init__(self):
        # Online and target networks
        self.online_net = DQNetwork().to(DEVICE)
        self.target_net = DQNetwork().to(DEVICE)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimiser  = optim.Adam(self.online_net.parameters(), lr=LEARNING_RATE)
        self.buffer     = ReplayBuffer(BUFFER_SIZE)
        self.steps_done = 0
        self.losses     = []

    def select_action(self, state, action_mask=None):
        """
        Epsilon-greedy action selection.
        Exploration decays from EPSILON_START to EPSILON_END over EPSILON_DECAY steps.
        """
        epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) *                   np.exp(-self.steps_done / EPSILON_DECAY)
        self.steps_done += 1

        if random.random() < epsilon:
            # Random valid action
            if action_mask is not None:
                valid = np.where(action_mask)[0]
                return int(random.choice(valid)) if len(valid) > 0 else 0
            return random.randint(0, MAX_TASKS - 1)
        else:
            # Greedy action
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                q_vals  = self.online_net(state_t).squeeze(0)
                if action_mask is not None:
                    mask_t = torch.BoolTensor(action_mask).to(DEVICE)
                    q_vals[~mask_t] = float('-inf')
                return int(q_vals.argmax().item())

    def store(self, state, action, reward, next_state, done):
        """Store transition in replay buffer."""
        self.buffer.push(state, action, reward, next_state, done)

    def train_step(self):
        """
        Sample mini-batch and perform one gradient update.
        Uses Double DQN loss for stability.
        """
        if len(self.buffer) < BATCH_SIZE:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample()

        # Current Q-values
        current_q = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q-values (Double DQN)
        with torch.no_grad():
            next_actions = self.online_net(next_states).argmax(1)
            next_q       = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            target_q     = rewards + GAMMA * next_q * (1 - dones)

        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), 1.0)
        self.optimiser.step()

        self.losses.append(loss.item())
        return loss.item()

    def update_target(self):
        """Copy online network weights to target network."""
        self.target_net.load_state_dict(self.online_net.state_dict())

    def current_epsilon(self):
        return EPSILON_END + (EPSILON_START - EPSILON_END) *                np.exp(-self.steps_done / EPSILON_DECAY)

    def save(self, path):
        torch.save({
            'online_net':  self.online_net.state_dict(),
            'target_net':  self.target_net.state_dict(),
            'steps_done':  self.steps_done,
            'losses':      self.losses,
        }, path)
        print(f"Model saved to {path}")

    def load(self, path):
        checkpoint = torch.load(path, map_location=DEVICE)
        self.online_net.load_state_dict(checkpoint['online_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.steps_done = checkpoint['steps_done']
        self.losses     = checkpoint['losses']
        print(f"Model loaded from {path}")


print("DQNAgent defined.")
print(f"Double DQN      : enabled")
print(f"Gradient clipping: 1.0")
print(f"Target update   : every {TARGET_UPDATE} episodes")


## 6. Training Loop

This cell runs the full 5,000-episode training run.

**Expected time:** ~1.5–2.5 hours on a CPU laptop.
Go make a coffee — or two.

Progress is printed every 100 episodes showing:
- Mean TCR over last 100 episodes
- Mean episode reward over last 100 episodes
- Current epsilon value
- Training loss


In [ ]:
env   = TaskSchedulerEnv(train_scenarios, seed=SEED)
agent = DQNAgent()

# ── Training history ──────────────────────────────────────────────────────────
history = {
    'episode':        [],
    'reward':         [],
    'tcr':            [],
    'dd':             [],
    'epsilon':        [],
    'loss':           [],
    'completed':      [],
    'missed':         [],
}

print(f"Starting training: {N_EPISODES} episodes")
print(f"Logging every     : {LOG_EVERY} episodes")
print(f"Model will be saved to: {MODEL_PATH}")
print()
print(f"{'Episode':>8} {'Reward':>10} {'TCR':>8} {'DD':>8} {'Epsilon':>9} {'Loss':>10}")
print("-" * 60)

start_time  = time.time()
window_size = 100
reward_window = deque(maxlen=window_size)
tcr_window    = deque(maxlen=window_size)
dd_window     = deque(maxlen=window_size)
loss_window   = deque(maxlen=window_size)

for episode in range(1, N_EPISODES + 1):
    state = env.reset()
    done  = False
    ep_reward = 0.0

    while not done:
        mask   = env.action_mask()
        action = agent.select_action(state, mask)
        next_state, reward, done, info = env.step(action)
        agent.store(state, action, reward, next_state, done)
        loss = agent.train_step()
        if loss is not None:
            loss_window.append(loss)
        state      = next_state
        ep_reward += reward

    # Update target network
    if episode % TARGET_UPDATE == 0:
        agent.update_target()

    # Record metrics
    metrics = env.get_metrics()
    reward_window.append(ep_reward)
    tcr_window.append(metrics['tcr'])
    dd_window.append(metrics['dd'])

    history['episode'].append(episode)
    history['reward'].append(ep_reward)
    history['tcr'].append(metrics['tcr'])
    history['dd'].append(metrics['dd'])
    history['epsilon'].append(agent.current_epsilon())
    history['loss'].append(np.mean(loss_window) if loss_window else 0.0)
    history['completed'].append(metrics['completed'])
    history['missed'].append(metrics['missed'])

    # Log progress
    if episode % LOG_EVERY == 0:
        mean_reward  = np.mean(reward_window)
        mean_tcr     = np.mean(tcr_window) * 100
        mean_dd      = np.mean(dd_window)
        mean_loss    = np.mean(loss_window) if loss_window else 0.0
        epsilon      = agent.current_epsilon()
        elapsed      = (time.time() - start_time) / 60
        eta          = (elapsed / episode) * (N_EPISODES - episode)
        print(f"{episode:>8} {mean_reward:>+10.1f} {mean_tcr:>7.1f}% "
              f"{mean_dd:>8.1f} {epsilon:>9.3f} {mean_loss:>10.4f}  "
              f"[{elapsed:.0f}m elapsed, ~{eta:.0f}m remaining]")

elapsed_total = (time.time() - start_time) / 60
print()
print(f"Training complete in {elapsed_total:.1f} minutes.")
print(f"Total steps: {agent.steps_done:,}")


## 7. Training Results and Charts

In [ ]:
df_hist = pd.DataFrame(history)
df_hist.to_csv(RESULTS_PATH, index=False)
print(f"Training history saved: {RESULTS_PATH}")

# ── Smoothing helper ──────────────────────────────────────────────────────────
def smooth(values, window=100):
    return pd.Series(values).rolling(window, min_periods=1).mean().values

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("DQN Training Results — 5,000 Episodes", fontsize=14, fontweight='bold')

ep = df_hist['episode'].values

# Plot 1: Episode reward
axes[0,0].plot(ep, df_hist['reward'], alpha=0.2, color=PALETTE[1])
axes[0,0].plot(ep, smooth(df_hist['reward']), color=PALETTE[1], linewidth=2, label='Smoothed (100ep)')
axes[0,0].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[0,0].set_title("Episode Reward", fontweight='bold')
axes[0,0].set_xlabel("Episode")
axes[0,0].set_ylabel("Total reward")
axes[0,0].legend()

# Plot 2: TCR over training
axes[0,1].plot(ep, df_hist['tcr']*100, alpha=0.2, color=PALETTE[0])
axes[0,1].plot(ep, smooth(df_hist['tcr']*100), color=PALETTE[0], linewidth=2, label='Smoothed (100ep)')
axes[0,1].axhline(38.5, color=PALETTE[4], linestyle='--', linewidth=1.5, label='Baseline TCR: 38.5%')
axes[0,1].set_title("Task Completion Rate (TCR)", fontweight='bold')
axes[0,1].set_xlabel("Episode")
axes[0,1].set_ylabel("TCR (%)")
axes[0,1].legend()

# Plot 3: Epsilon decay
axes[0,2].plot(ep, df_hist['epsilon'], color=PALETTE[2], linewidth=2)
axes[0,2].set_title("Epsilon Decay (Exploration Rate)", fontweight='bold')
axes[0,2].set_xlabel("Episode")
axes[0,2].set_ylabel("Epsilon")
axes[0,2].fill_between(ep, df_hist['epsilon'], alpha=0.2, color=PALETTE[2])

# Plot 4: Training loss
axes[1,0].plot(ep, df_hist['loss'], alpha=0.3, color=PALETTE[3])
axes[1,0].plot(ep, smooth(df_hist['loss']), color=PALETTE[3], linewidth=2, label='Smoothed')
axes[1,0].set_title("Training Loss (Huber)", fontweight='bold')
axes[1,0].set_xlabel("Episode")
axes[1,0].set_ylabel("Loss")
axes[1,0].legend()

# Plot 5: Deadline deviation
axes[1,1].plot(ep, df_hist['dd'], alpha=0.2, color=PALETTE[4])
axes[1,1].plot(ep, smooth(df_hist['dd']), color=PALETTE[4], linewidth=2, label='Smoothed')
axes[1,1].axhline(116.4, color='gray', linestyle='--', linewidth=1.5, label='FIFO baseline')
axes[1,1].set_title("Mean Deadline Deviation", fontweight='bold')
axes[1,1].set_xlabel("Episode")
axes[1,1].set_ylabel("Minutes")
axes[1,1].legend()

# Plot 6: Completed vs missed
axes[1,2].plot(ep, smooth(df_hist['completed']), color=PALETTE[0], linewidth=2, label='Completed')
axes[1,2].plot(ep, smooth(df_hist['missed']),    color=PALETTE[4], linewidth=2, label='Missed')
axes[1,2].set_title("Tasks Completed vs Missed", fontweight='bold')
axes[1,2].set_xlabel("Episode")
axes[1,2].set_ylabel("Tasks (smoothed)")
axes[1,2].legend()

plt.tight_layout()
plt.savefig("data/fig9_training_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: data/fig9_training_results.png")


## 8. Save Trained Model

In [ ]:
agent.save(MODEL_PATH)

# ── Print final training statistics ──────────────────────────────────────────
last_500 = df_hist.tail(500)
print("=" * 55)
print("  TRAINING COMPLETE")
print("=" * 55)
print(f"  Total episodes     : {N_EPISODES:,}")
print(f"  Total steps        : {agent.steps_done:,}")
print()
print("  Performance (last 500 episodes):")
print(f"  Mean TCR           : {last_500['tcr'].mean()*100:.2f}%")
print(f"  Mean DD            : {last_500['dd'].mean():.2f} min")
print(f"  Mean reward        : {last_500['reward'].mean():.2f}")
print(f"  Final epsilon      : {agent.current_epsilon():.4f}")
print()
print("  Comparison:")
print(f"  Random agent TCR   : ~30.0%")
print(f"  Baseline TCR       : ~38.5%")
print(f"  DQN TCR (last 500) : {last_500['tcr'].mean()*100:.2f}%")
print("=" * 55)


## 9. Quick Evaluation on Test Set

Run the trained agent on all 200 test scenarios to get preliminary
evaluation numbers. Full comparative evaluation is in notebook 06.


In [ ]:
print("Evaluating trained DQN on 200 test scenarios...")

agent.online_net.eval()
test_env = TaskSchedulerEnv(test_scenarios, seed=SEED)
test_results = []

for scenario in test_scenarios:
    test_env_s = TaskSchedulerEnv([scenario], seed=SEED)
    state      = test_env_s.reset()
    done       = False

    while not done:
        mask   = test_env_s.action_mask()
        # Greedy (epsilon=0 for evaluation)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            q_vals  = agent.online_net(state_t).squeeze(0)
            mask_t  = torch.BoolTensor(mask).to(DEVICE)
            q_vals[~mask_t] = float('-inf')
            action  = int(q_vals.argmax().item())
        state, _, done, _ = test_env_s.step(action)

    m = test_env_s.get_metrics()
    test_results.append({
        'scenario_id': scenario['scenario_id'],
        'complexity':  scenario['complexity'],
        'tcr':         m['tcr'],
        'dd':          m['dd'],
        'reward':      m['episode_reward'],
        'completed':   m['completed'],
        'missed':      m['missed'],
    })

df_test = pd.DataFrame(test_results)
df_test.to_csv("data/dqn_test_results.csv", index=False)

print()
print("=" * 55)
print("  DQN TEST SET RESULTS (200 scenarios)")
print("=" * 55)
print(f"  TCR              : {df_test['tcr'].mean()*100:.2f}%")
print(f"  Mean DD          : {df_test['dd'].mean():.2f} min")
print(f"  Mean reward      : {df_test['reward'].mean():.2f}")
print()
print("  By complexity:")
for c in ['low', 'medium', 'high']:
    s = df_test[df_test['complexity'] == c]
    print(f"    {c:<8} TCR={s['tcr'].mean()*100:.1f}%  DD={s['dd'].mean():.1f}min")
print()
print("  vs Baselines:")
print(f"  FIFO TCR         : 38.49%")
print(f"  Priority Q TCR   : 37.79%")
print(f"  DQN TCR          : {df_test['tcr'].mean()*100:.2f}%")
improvement = df_test['tcr'].mean()*100 - 38.49
print(f"  Improvement      : {improvement:+.2f} pp vs FIFO")
print("=" * 55)


## 10. Training Summary

In [ ]:
print("=" * 55)
print("  NOTEBOOK 05 COMPLETE")
print("=" * 55)
print()
print("  Files saved:")
for f in [MODEL_PATH, RESULTS_PATH,
          "data/dqn_test_results.csv",
          "data/fig9_training_results.png"]:
    size = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"    {f:<40} {size:.1f} KB")
print()
print("  Model architecture:")
print(f"    {STATE_SIZE} -> {HIDDEN_1} -> {HIDDEN_2} -> {HIDDEN_3} -> {MAX_TASKS}")
print(f"    Parameters: {sum(p.numel() for p in agent.online_net.parameters()):,}")
print()
print("  Training config:")
print(f"    Episodes        : {N_EPISODES:,}")
print(f"    Steps           : {agent.steps_done:,}")
print(f"    Buffer size     : {BUFFER_SIZE:,}")
print(f"    Batch size      : {BATCH_SIZE}")
print(f"    Final epsilon   : {agent.current_epsilon():.4f}")
print()
print("  Next step → 06_Evaluation.ipynb")
print("=" * 55)
